# Persiapan Docking

**Import Library**

In [29]:
import os
import requests

import MDAnalysis as mda
import nglview as nv
import numpy as np

from rcsbapi.search import search_attributes as attrs


## Mencari Protein Berdasarkan Organisme Sumber

### Cari Protein

In [30]:
ORGANISM = "colletotrichum gloeosporioides"

query = attrs.rcsb_entity_source_organism.taxonomy_lineage.name == ORGANISM
query_result = list(query())
query_result

['3DCN', '3DD5', '3DEA']

### Simpan Protein

In [31]:
pdb_id = query_result[1].lower()

In [32]:
PROTEIN_DIR = "proteins"
os.makedirs(PROTEIN_DIR, exist_ok=True)

In [33]:
pdb_request = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb")
if pdb_request.status_code != 200:
    print("Error download protein")

In [34]:
with open(f"{PROTEIN_DIR}/{pdb_id}.pdb", "w+") as file:
    file.write(pdb_request.text)

## Visualisasi Struktur Protein

In [35]:
u = mda.Universe(f"{PROTEIN_DIR}/{pdb_id}.pdb")

### Residu

In [36]:
np.unique(np.array(u.residues.resnames))

array(['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'DEP', 'GLN', 'GLU', 'GLY',
       'HIS', 'HOH', 'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER',
       'THR', 'TRP', 'TYR', 'VAL'], dtype=object)

In [37]:
RESIDUE_NAME = "DEP"

In [38]:
view = nv.show_mdanalysis(u)
view

NGLWidget()

In [39]:
protein = u.select_atoms("protein")
ligand = u.select_atoms(f"resname {RESIDUE_NAME}")
water = u.select_atoms("resname HOH")


In [40]:
nv.show_mdanalysis(ligand)

NGLWidget()

In [41]:
# Simpan protein
protein.write(filename=f"{PROTEIN_DIR}/protein_{pdb_id}.pdb")

/home/fynn/Projects/molecular/venv/lib64/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:1154: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn("Found no information for attr: '{}'"


## Perbaikan Struktur Protein

In [42]:
! pdb2pqr --pdb-output=proteins/protein_h.pdb --pH=7.4 proteins/protein_3dd5.pdb proteins/protein_3dd5.pqr --whitespace

INFO:PDB2PQR v3.6.2: biomolecular structure conversion software.
INFO:Please cite:  Jurrus E, et al.  Improvements to the APBS biomolecular solvation software suite.  Protein Sci 27 112-128 (2018).
INFO:Please cite:  Dolinsky TJ, et al.  PDB2PQR: expanding and upgrading automated preparation of biomolecular structures for molecular simulations. Nucleic Acids Res 35 W522-W525 (2007).
INFO:Checking and transforming input arguments.
INFO:Loading topology files.
INFO:Loading molecule: proteins/protein_3dd5.pdb
ERROR:Error parsing line: invalid literal for int() with base 10: ''
ERROR:<REMARK     2>
ERROR:Truncating remaining errors for record type:REMARK

ERROR:['REMARK']
INFO:Setting up molecule.
INFO:Created biomolecule object with 1547 residues and 11428 atoms.
INFO:Setting termini states for biomolecule chains.
INFO:Loading forcefield.
INFO:Loading hydrogen topology definitions.
INFO:Attempting to repair 7 missing atoms in biomolecule.
INFO:Added atom OG1 to residue THR A 58 at coordin

## Membuat PDBQT File

In [43]:
PDBQT_DIR = "pdbqt"
os.makedirs(PDBQT_DIR, exist_ok=True)

In [44]:
u = mda.Universe(f"{PROTEIN_DIR}/protein_{pdb_id}.pqr")
u.atoms.write(f"{PDBQT_DIR}/{pdb_id}.pdbqt")

/home/fynn/Projects/molecular/venv/lib64/python3.10/site-packages/MDAnalysis/coordinates/PDBQT.py:305: UserWarning: Supplied AtomGroup was missing the following attributes: altLocs, occupancies, tempfactors. These will be written with default values. 
  warnings.warn(


In [45]:
# Read in the just-written PDBQT file, replace text, and write back
with open(f"{PDBQT_DIR}/{pdb_id}.pdbqt", 'r') as file:
    file_content = file.read()

# Replace 'TITLE' and 'CRYST1' with 'REMARK'
file_content = file_content.replace('TITLE', 'REMARK').replace('CRYST1', 'REMARK')

# Write the modified content back to the file
with open(f"{PDBQT_DIR}/{pdb_id}.pdbqt", 'w') as file:
    file.write(file_content)

## Mencari Ligan


In [46]:
resDEP_sdf = requests.get('https://files.rcsb.org/ligands/download/DEP_ideal.sdf')

In [47]:

LIGANDS_DIR = "ligands"
os.makedirs(LIGANDS_DIR, exist_ok=True)

In [48]:
with open(f"{LIGANDS_DIR}/dep.sdf", "w") as f:
    f.write(resDEP_sdf.text)

In [49]:
# Use meeko to prepare small molecules - using meeko helps us visualize them later.
! mk_prepare_ligand.py -i ligands/dep.sdf -o pdbqt/dep.pdbqt

[RDKit] WARNING:[17:08:40] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0
